# Classificação: modelos lineares, vizinhos, margem e árvores

**Para quem está começando.** Nenhuma linha de código aqui assume que você já viu classificação. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. entender o que muda entre quatro modelos clássicos;
2. ver por que a escala dos atributos muda o resultado de alguns deles e não de outros;
3. comparar os quatro na mesma divisão, com validação cruzada;
4. olhar a fronteira de decisão que cada um desenha.

Roda no Google Colab ou no Jupyter do seu computador. Só precisa de `numpy`, `pandas`, `scikit-learn` e `matplotlib`, que já vêm no Colab.


## Quatro modelos, em uma frase cada

Todos resolvem o mesmo problema (dizer se o tumor é maligno ou benigno), e todos foram inventados em décadas diferentes. O que muda entre eles é **a forma da fronteira** que conseguem desenhar.

- **Regressão logística**: combina os atributos numa soma e passa o resultado por uma curva que devolve probabilidade. A fronteira é uma reta (ou um plano). É o modelo que você deve tentar primeiro, porque é rápido e difícil de errar.
- **k-vizinhos mais próximos (kNN)**: não aprende nada no treino. Guarda os exemplos e, para classificar um ponto novo, olha os `k` vizinhos mais próximos e vota. A fronteira é irregular, e o `k` decide o quanto.
- **Máquina de vetores de suporte (SVM)**: procura a fronteira que deixa a maior margem entre as classes. Com kernel, ela pode ser curva. É forte em tabelas pequenas.
- **Árvore de decisão**: aprende regras do tipo "se esta medida for maior que tal valor, então...". A fronteira é feita de retângulos, e a profundidade máxima decide o quanto ela se dobra ao dado.

Um detalhe que vale para os dois do meio: kNN e SVM medem **distância** entre pontos. É por isso que a escala dos atributos, que é o primeiro passo deste notebook, muda o resultado deles.


## Passo 0: ambiente e configuração

Nada para instalar além do que já vem no Colab. A semente é a mesma em todas as células, então duas execuções dão o mesmo resultado.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (9, 4)

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=RANDOM_STATE,
)
print(f"✅ {X_train.shape[0]} linhas de treino | {X_test.shape[0]} de teste | "
      f"{X_train.shape[1]} atributos | {np.unique(y_train).size} classes")

## Passo 1: a escala muda o resultado

Os 30 atributos desta tabela não têm a mesma unidade nem a mesma ordem de grandeza. Um deles é uma área, na casa dos milhares; outro é uma suavidade, perto de 0,1.

Para um modelo que mede distância, isso é um problema: a diferença de 1000 na área domina qualquer diferença no outro atributo, e o modelo passa a decidir quase só pela área. A célula abaixo mede o efeito. O `StandardScaler` dentro do `make_pipeline` é a correção: ele padroniza cada coluna antes de o modelo ver o dado.


In [ ]:
modelos = {
    "kNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (rbf)": SVC(kernel="rbf"),
    "Regressão logística": LogisticRegression(max_iter=5000),
    "Árvore (prof. 4)": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
}

linhas = []
for nome, modelo in modelos.items():
    sem_escala = modelo.fit(X_train, y_train).score(X_test, y_test)
    com_escala = make_pipeline(StandardScaler(), modelo).fit(X_train, y_train).score(X_test, y_test)
    linhas.append({"modelo": nome, "sem escala": sem_escala, "com escala": com_escala,
                   "diferença": com_escala - sem_escala})

tabela_escala = pd.DataFrame(linhas).set_index("modelo")
display(tabela_escala.round(4))

### Como ler isso

O kNN e o SVM são os que mais ganham com a padronização, porque dependem de distância para decidir. A árvore não muda, porque ela compara cada atributo com um limiar dentro do próprio atributo: multiplicar uma coluna por mil não muda a ordem dos valores.

Daí a regra prática: padronize sempre que o modelo mede distância (kNN, SVM, redes neurais, k-means); a árvore e os modelos de boosting não precisam, mas também não se incomodam.

Repare que a regressão logística fica no meio: ela tem uma soma ponderada, então a escala muda o tamanho dos coeficientes e o efeito da penalização, mas ela consegue compensar mexendo nos pesos.


## Passo 2: os quatro modelos, na mesma régua

Agora a comparação honesta. Uma divisão só de treino e teste não decide nada, então cada modelo é avaliado com validação cruzada de 5 folds, e o resultado sai com média e desvio. Duas métricas: acurácia (quantos acertos) e ROC-AUC (o quão bem o modelo ordena os exemplos por probabilidade). Tempo de treino e de predição entram na tabela porque, em muitos casos, é o que decide qual modelo vai para produção.


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def timeit(fn, *args, **kwargs):
    """Run fn once and return (result, elapsed_seconds)."""
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - start


linhas = []
for nome, modelo in modelos.items():
    pipeline = make_pipeline(StandardScaler(), modelo)
    _, fit_s = timeit(pipeline.fit, X_train, y_train)
    _, predict_s = timeit(pipeline.predict, X_test)
    acuracia = cross_val_score(pipeline, X_train, y_train, cv=folds, scoring="accuracy")
    auc = cross_val_score(pipeline, X_train, y_train, cv=folds, scoring="roc_auc")
    linhas.append({"modelo": nome, "acuracia_cv": acuracia.mean(), "desvio": acuracia.std(),
                   "roc_auc_cv": auc.mean(), "fit_s": fit_s, "predict_s": predict_s})

tabela = pd.DataFrame(linhas).set_index("modelo").sort_values("roc_auc_cv", ascending=False)
display(tabela.round(4))

### Como ler isso

A regressão logística, a mais simples das quatro, sai na frente em acurácia. Isso não é acidente: a tabela é pequena, bem separada, e uma fronteira reta já resolve quase tudo. Modelo mais complicado não é modelo melhor, e é por isso que a comparação começa pelo mais simples. Olhe três colunas por vez:

- **`desvio`**: é o quanto a métrica muda de fold para fold. Se dois modelos diferem menos que isso, a diferença não é resultado, é sorte da partição. É o cuidado que vale para todo este repositório;
- **`roc_auc_cv`** contra **`acuracia_cv`**: acurácia conta acertos, AUC mede ordenação. Com classes desbalanceadas, a acurácia engana, e a AUC continua informando;
- **`fit_s`** e **`predict_s`**: o kNN é disparado o mais lento para prever, porque faz uma busca por vizinhos a cada predição, ainda que o treino dele seja instantâneo. Com 426 linhas de treino os tempos são todos pequenos, e é justamente essa coluna que passa a decidir quando a tabela cresce.


## Passo 3: a fronteira de decisão de cada um

Números não mostram bem o que muda entre os modelos. Para ver, dá para usar só dois atributos e desenhar a região que cada modelo atribui a cada classe. É um recorte didático: a acurácia no título é medida sobre esses dois atributos, então não é comparável à da tabela anterior, mas a forma da fronteira é a mesma que cada modelo teria com os 30.


In [ ]:
colunas = [0, 1]          # mean radius e mean texture
X2, y2 = X_train[:, colunas], y_train

fig, eixos = plt.subplots(1, 4, figsize=(18, 4.2), sharey=True)
for eixo, (nome, modelo) in zip(eixos, modelos.items()):
    pipeline = make_pipeline(StandardScaler(), modelo).fit(X2, y2)
    DecisionBoundaryDisplay.from_estimator(pipeline, X2, ax=eixo, alpha=0.3, cmap="coolwarm")
    eixo.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="coolwarm", edgecolor="k", s=12)
    eixo.set_title(f"{nome}\nacuracia {pipeline.score(X2, y2):.3f}")
    eixo.set_xlabel(data.feature_names[colunas[0]])
eixos[0].set_ylabel(data.feature_names[colunas[1]])
plt.tight_layout()
plt.show()

### Como ler isso

As quatro fronteiras têm formas diferentes, e isso não é acidente: cada modelo tem uma família de formas que consegue representar. A regressão logística só sabe desenhar uma reta. O kNN desenha ilhas em volta dos vizinhos, então com `k` pequeno ele decora o ruído. O SVM com kernel rbf desenha curvas suaves, o que costuma generalizar melhor que o kNN nesta tabela. A árvore desenha degraus, e cada degrau é uma regra do tipo "se... então".

Duas coisas para experimentar depois: mude `k` de 5 para 1 e para 50 e veja a fronteira do kNN entre decorar e suavizar demais; e troque o kernel do SVM para `linear` para ver a fronteira voltar a ser reta.

## Resumo

Modelos clássicos não são um só: o que os separa é a família de fronteiras que cada um representa e o custo de treinar e prever. Antes de escolher, dois passos que mudam o resultado mais que a escolha do modelo: padronizar os atributos quando o modelo mede distância, e medir com validação cruzada em vez de confiar numa divisão só.

**Próximo passo:** o notebook de regressão da mesma pasta, onde o alvo é um número em vez de uma classe, e a árvore de decisão aparece de novo, agora com o problema de decorar o treino.

### Referências

**Os modelos**

Cover, T., & Hart, P. (1967). *Nearest neighbor pattern classification.* IEEE Transactions on Information Theory 13(1), 21-27.

Cortes, C., & Vapnik, V. (1995). *Support-vector networks.* Machine Learning 20(3), 273-297.

Breiman, L., Friedman, J., Olshen, R., & Stone, C. (1984). *Classification and Regression Trees.* Wadsworth.

**O contexto**

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. Capítulo 7, sobre a maldição da dimensionalidade e a necessidade de padronizar.

```bibtex
@article{cover1967nearest,
  title   = {Nearest neighbor pattern classification},
  author  = {Cover, Thomas and Hart, Peter},
  journal = {IEEE Transactions on Information Theory},
  volume  = {13},
  number  = {1},
  pages   = {21--27},
  year    = {1967}
}

@article{cortes1995support,
  title   = {Support-vector networks},
  author  = {Cortes, Corinna and Vapnik, Vladimir},
  journal = {Machine Learning},
  volume  = {20},
  number  = {3},
  pages   = {273--297},
  year    = {1995}
}

@book{breiman1984classification,
  title     = {Classification and Regression Trees},
  author    = {Breiman, Leo and Friedman, Jerome and Olshen, Richard and Stone, Charles},
  year      = {1984},
  publisher = {Wadsworth}
}

@book{hastie2009elements,
  title     = {The Elements of Statistical Learning},
  author    = {Hastie, Trevor and Tibshirani, Robert and Friedman, Jerome},
  year      = {2009},
  edition   = {2},
  publisher = {Springer}
}
```
